In [3]:
!pip install torcheval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 5.3 MB/s eta 0:00:00


In [35]:
from google.colab import files
uploaded = files.upload()

Saving best_model.pth to best_model.pth


In [36]:
import os, shutil

os.makedirs("results_dnn", exist_ok=True)
shutil.move("best_model.pth", "results_dnn/best_model.pth")
print("DNN model placed in results_dnn/")

DNN model placed in results_dnn/


In [42]:
!ls results_dnn

best_model.pth


In [47]:
from google.colab import files
uploaded = files.upload()

Saving best_model.pth to best_model.pth


In [48]:
import os, shutil

os.makedirs("results_cnn", exist_ok=True)
shutil.move("best_model.pth", "results_cnn/best_model.pth")

print("CNN model successfully moved to results_cnn/")

CNN model successfully moved to results_cnn/


In [52]:
from google.colab import files
uploaded = files.upload()

Saving best_model.pth to best_model.pth


In [53]:
import os, shutil

os.makedirs("results_vgg", exist_ok=True)
shutil.move("best_model.pth", "results_vgg/best_model.pth")

print("VGG model successfully moved to results_vgg/")

VGG model successfully moved to results_vgg/


In [54]:
from google.colab import files
uploaded = files.upload()

Saving best_model.pth to best_model.pth


In [55]:
import os, shutil

os.makedirs("results_resnet", exist_ok=True)
shutil.move("best_model.pth", "results_resnet/best_model.pth")

print("ResNet model successfully moved to results_resnet/")

ResNet model successfully moved to results_resnet/


In [61]:
from google.colab import files
uploaded = files.upload()

Saving cnn_model.py to cnn_model.py
Saving dnn_model.py to dnn_model.py
Saving resnet_model.py to resnet_model.py
Saving vgg_model.py to vgg_model.py


In [62]:
!ls /content

cnn_model.py  __pycache__      results_cnn  results_resnet  sample_data
dnn_model.py  resnet_model.py  results_dnn  results_vgg     vgg_model.py


In [63]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import roc_auc_score
from torcheval.metrics.functional import multiclass_f1_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_dataset = datasets.MNIST("../data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

loss_fn = torch.nn.CrossEntropyLoss()

def evaluate(model, loss_fn, device, dataloader):
    model.eval()
    total_loss, correct = 0, 0
    f1_scores, auc_scores = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            f1 = multiclass_f1_score(preds, labels, num_classes=10)
            f1_scores.append(f1.item())
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            try:
                auc = roc_auc_score(labels.cpu().numpy(), probs, multi_class="ovr")
                auc_scores.append(auc)
            except Exception:
                pass
    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / len(dataloader.dataset)
    avg_f1 = np.mean(f1_scores)
    avg_auc = np.mean(auc_scores) if auc_scores else 0.0
    return avg_loss, accuracy, avg_f1, avg_auc


In [64]:
# Evaluate DNN

from dnn_model import DNNModel

model_name = "dnn"
model = DNNModel().to(device)
model.load_state_dict(torch.load(f"./results_{model_name}/best_model.pth", map_location=device))

test_loss, test_acc, test_f1, test_auc = evaluate(model, loss_fn, device, test_loader)
print(f"[{model_name.upper()}] Accuracy: {test_acc:.2f}% | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

with open(f"./results_{model_name}/test_results.txt", "w") as f:
    f.write(f"Accuracy: {test_acc:.2f}%\nF1: {test_f1:.4f}\nAUC: {test_auc:.4f}\n")


[DNN] Accuracy: 97.16% | F1: 0.9719 | AUC: 0.9992


In [65]:
# Evaluate CNN

from cnn_model import CNNModel

model_name = "cnn"
model = CNNModel().to(device)
model.load_state_dict(torch.load(f"./results_{model_name}/best_model.pth", map_location=device))

test_loss, test_acc, test_f1, test_auc = evaluate(model, loss_fn, device, test_loader)
print(f"[{model_name.upper()}] Accuracy: {test_acc:.2f}% | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

[CNN] Accuracy: 99.05% | F1: 0.9906 | AUC: 0.9999


In [66]:
# Evaluate VGG

from vgg_model import VGGModel

model_name = "vgg"
model = VGGModel().to(device)
model.load_state_dict(torch.load(f"./results_{model_name}/best_model.pth", map_location=device))

test_loss, test_acc, test_f1, test_auc = evaluate(model, loss_fn, device, test_loader)
print(f"[{model_name.upper()}] Accuracy: {test_acc:.2f}% | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

with open(f"./results_{model_name}/test_results.txt", "w") as f:
    f.write(f"Accuracy: {test_acc:.2f}%\nF1: {test_f1:.4f}\nAUC: {test_auc:.4f}\n")


[VGG] Accuracy: 99.26% | F1: 0.9927 | AUC: 0.9999


In [67]:
# Evaluate RESNET

from resnet_model import ResNetModel

model_name = "resnet"
model = ResNetModel().to(device)
model.load_state_dict(torch.load(f"./results_{model_name}/best_model.pth", map_location=device))

test_loss, test_acc, test_f1, test_auc = evaluate(model, loss_fn, device, test_loader)
print(f"[{model_name.upper()}] Accuracy: {test_acc:.2f}% | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

with open(f"./results_{model_name}/test_results.txt", "w") as f:
    f.write(f"Accuracy: {test_acc:.2f}%\nF1: {test_f1:.4f}\nAUC: {test_auc:.4f}\n")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


[RESNET] Accuracy: 80.71% | F1: 0.8092 | AUC: 0.9844
